# Embeddings Analysis
This notebook covers:
1. Training curves (loss + norm)
2. Genre family visualisation
3. Genre cluster visualisation
4. All genres labelled
5. Intra-cluster vs global distance validation

## 1. Training Curves

In [ ]:
import re
import matplotlib.pyplot as plt

# ── config ────────────────────────────────────────────────────────────────────
LOG_FILE = 'training_log.txt'
# ─────────────────────────────────────────────────────────────────────────────

losses, norms = [], []
with open(LOG_FILE) as f:
    for line in f:
        m = re.search(r'Epoch \d+ \| loss: ([\d.]+) \| embedding norm mean: ([\d.]+)', line)
        if m:
            losses.append(float(m.group(1)))
            norms.append(float(m.group(2)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(losses)
ax1.set_title('Loss per epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')

ax2.plot(norms, color='orange')
ax2.set_title('Embedding norm mean per epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Norm')

plt.tight_layout()
# plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')  # uncomment to save
plt.show()

## Setup — Load Model (shared by all plot cells below)

In [ ]:
import torch
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from lorentz import Lorentz

# ── config ────────────────────────────────────────────────────────────────────
CKPT   = 'ckpt/final_enoa_graph.ckpt'
VOCAB  = 'enoa_vocab.pkl'
MATRIX = 'enoa_graph.npy'
DIM    = 2
# ─────────────────────────────────────────────────────────────────────────────

genres = pickle.load(open(VOCAB, 'rb'))
n_items = len(genres)
adj = np.load(MATRIX)
genre_to_idx = {g: i for i, g in enumerate(genres)}

net = Lorentz(n_items, DIM + 1)
net.load_state_dict(torch.load(CKPT, map_location='cpu'))
net.eval()

table = net.lorentz_to_poincare()
coords = table[1:]
x = coords[:, 0]
y = coords[:, 1]

# also get lorentz embeddings for distance computation
lorentz_table = net.table.weight.data.cpu().numpy()

def lorentz_distance(u, v):
    inner = -u[0]*v[0] + np.dot(u[1:], v[1:])
    inner = np.clip(-inner, 1 + 1e-6, None)
    return float(np.arccosh(inner))

def get_embedding(genre_name):
    if genre_name not in genre_to_idx:
        return None
    idx = genre_to_idx[genre_name]
    return lorentz_table[idx + 1]

print(f'Loaded {n_items} genres and embeddings')

## 2. Genre Families Plot

In [ ]:
FIGSIZE  = (20, 20)
FONTSIZE = 9

FAMILIES = {
    'Latin':      {'genres': ['latin', 'reggaeton', 'cumbia', 'bachata', 'latin pop',
                               'trap latino', 'latin hip hop', 'reggaeton flow'],
                   'color': '#e63946'},
    'Hip Hop':    {'genres': ['rap', 'trap', 'hip hop', 'pop rap', 'melodic rap',
                               'gangster rap', 'east coast hip hop', 'southern hip hop'],
                   'color': '#f4a261'},
    'Rock':       {'genres': ['rock', 'metal', 'blues rock', 'indie rock', 'hard rock',
                               'alternative rock', 'classic rock', 'punk'],
                   'color': '#2a9d8f'},
    'Electronic': {'genres': ['edm', 'house', 'electropop', 'progressive house',
                               'tropical house', 'deep house', 'techno', 'dance pop'],
                   'color': '#457b9d'},
    'Pop':        {'genres': ['pop', 'dance pop', 'post-teen pop', 'indie pop',
                               'art pop', 'synthpop', 'k-pop', 'j-pop'],
                   'color': '#a8dadc'},
}

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
ax.add_artist(plt.Circle((0, 0), 1, fill=False, edgecolor='black', linewidth=1.5))
ax.scatter(x, y, s=8, color='lightgrey', zorder=2, linewidths=0)

legend_patches = []
for family_name, info in FAMILIES.items():
    color = info['color']
    found_indices = [genre_to_idx[g] for g in info['genres'] if g in genre_to_idx]
    found_genres  = [g for g in info['genres'] if g in genre_to_idx]
    missing = [g for g in info['genres'] if g not in genre_to_idx]
    if missing:
        print(f'{family_name} — missing: {missing}')
    if not found_indices:
        continue
    ax.scatter([x[i] for i in found_indices], [y[i] for i in found_indices],
               s=60, color=color, zorder=4, linewidths=0)
    for i, genre in zip(found_indices, found_genres):
        ax.annotate(genre, (x[i], y[i]), fontsize=FONTSIZE, ha='center', va='bottom',
                    xytext=(0, 5), textcoords='offset points', zorder=5,
                    color=color, fontweight='bold',
                    bbox={'fc': 'white', 'alpha': 0.85, 'edgecolor': 'none', 'pad': 1})
    legend_patches.append(mpatches.Patch(color=color, label=family_name))

ax.legend(handles=legend_patches, loc='lower right', fontsize=12, framealpha=0.9)
ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.05, 1.05)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Poincaré Disk — Genre Families', fontsize=16, pad=15)
plt.tight_layout()
# plt.savefig('plot_genre_families.png', dpi=200, bbox_inches='tight')  # uncomment to save
plt.show()

## 3. Genre Clusters Plot

In [ ]:
FIGSIZE   = (20, 20)
FONTSIZE  = 10
DOT_SIZE  = 80
GREY_SIZE = 6

CLUSTERS = {
    'Pop': {
        'genres': ['pop', 'dance pop', 'canadian pop', 'latin pop', 'j-pop', 'k-pop',
                   'indie pop', 'bedroom pop', 'post-teen pop', 'electropop'],
        'color': '#e63946'
    },
    'Hip Hop / Rap': {
        'genres': ['rap', 'hip hop', 'trap', 'pop rap', 'melodic rap', 'atl hip hop',
                   'southern hip hop', 'french hip hop', 'gangster rap', 'conscious hip hop',
                   'east coast hip hop', 'west coast rap', 'cloud rap'],
        'color': '#f4a261'
    },
    'Rock': {
        'genres': ['rock', 'modern rock', 'classic rock', 'alternative rock', 'hard rock',
                   'post-grunge', 'pop rock', 'indie rock', 'folk rock', 'psychedelic rock',
                   'garage rock', 'art rock'],
        'color': '#2a9d8f'
    },
    'Latin': {
        'genres': ['urbano latino', 'trap latino', 'reggaeton', 'musica mexicana', 'corrido',
                   'norteno', 'banda', 'reggaeton colombiano', 'latin alternative',
                   'latin rock', 'latin hip hop', 'ranchera'],
        'color': '#8338ec'
    },
    'Electronic / Dance': {
        'genres': ['edm', 'electro house', 'house', 'progressive house', 'tropical house',
                   'slap house', 'progressive electro house', 'techno', 'deep house'],
        'color': '#457b9d'
    },
}

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
ax.add_artist(plt.Circle((0, 0), 1, fill=False, edgecolor='black', linewidth=1.5))
ax.scatter(x, y, s=GREY_SIZE, color='lightgrey', zorder=2, linewidths=0)

legend_patches = []
for cluster_name, info in CLUSTERS.items():
    color = info['color']
    found_indices = [genre_to_idx[g] for g in info['genres'] if g in genre_to_idx]
    found_genres  = [g for g in info['genres'] if g in genre_to_idx]
    if not found_indices:
        continue
    ax.scatter([x[i] for i in found_indices], [y[i] for i in found_indices],
               s=DOT_SIZE, color=color, zorder=4, linewidths=0)
    for i, genre in zip(found_indices, found_genres):
        ax.annotate(genre, (x[i], y[i]), fontsize=FONTSIZE, ha='center', va='bottom',
                    xytext=(0, 6), textcoords='offset points', zorder=5,
                    color=color, fontweight='bold',
                    bbox={'fc': 'white', 'alpha': 0.85, 'edgecolor': 'none', 'pad': 1})
    legend_patches.append(mpatches.Patch(color=color, label=cluster_name))

ax.legend(handles=legend_patches, loc='lower right', fontsize=12, framealpha=0.9)
ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.05, 1.05)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Poincaré Disk — Genre Clusters', fontsize=16, pad=15)
plt.tight_layout()
# plt.savefig('plot_clusters.png', dpi=200, bbox_inches='tight')  # uncomment to save
plt.show()

## 4. All Genres Labelled

In [ ]:
FIGSIZE  = (60, 60)
FONTSIZE = 4

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
ax.add_artist(plt.Circle((0, 0), 1, fill=False, edgecolor='black', linewidth=1.5))
ax.scatter(x, y, s=5, color='tomato', zorder=2, linewidths=0)

for idx, genre in enumerate(genres):
    ax.annotate(genre, (x[idx], y[idx]), fontsize=FONTSIZE, ha='center', va='bottom',
                xytext=(0, 2), textcoords='offset points', zorder=3,
                bbox={'fc': 'white', 'alpha': 0.6, 'edgecolor': 'none', 'pad': 0.5})

ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.05, 1.05)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Poincaré Disk — All Genres', fontsize=24, pad=15)
plt.tight_layout()
# plt.savefig('plot_all_genres.png', dpi=150, bbox_inches='tight')  # uncomment to save
plt.show()

## 5. Intra-cluster vs Global Distance Validation

**Idea:** If the embeddings are correct, genres within the same cluster should be closer to each other than the global average distance between all genre pairs. We compute:
- **Global average distance** — mean Lorentz distance over a random sample of genre pairs
- **Intra-cluster average distance** — mean distance between all pairs within each cluster

If the embeddings are meaningful, intra-cluster distances should be significantly smaller than the global average.

In [ ]:
import itertools
import pandas as pd

# use the same clusters defined above
CLUSTERS_FOR_VALIDATION = {
    'Pop':           ['pop', 'dance pop', 'canadian pop', 'latin pop', 'j-pop', 'k-pop',
                      'indie pop', 'bedroom pop', 'post-teen pop', 'electropop'],
    'Hip Hop / Rap': ['rap', 'hip hop', 'trap', 'pop rap', 'melodic rap', 'atl hip hop',
                      'southern hip hop', 'french hip hop', 'gangster rap', 'conscious hip hop',
                      'east coast hip hop', 'west coast rap', 'cloud rap'],
    'Rock':          ['rock', 'modern rock', 'classic rock', 'alternative rock', 'hard rock',
                      'post-grunge', 'pop rock', 'indie rock', 'folk rock', 'psychedelic rock',
                      'garage rock', 'art rock'],
    'Latin':         ['urbano latino', 'trap latino', 'reggaeton', 'musica mexicana', 'corrido',
                      'norteno', 'banda', 'reggaeton colombiano', 'latin alternative',
                      'latin rock', 'latin hip hop', 'ranchera'],
    'Electronic':    ['edm', 'electro house', 'house', 'progressive house', 'tropical house',
                      'slap house', 'progressive electro house', 'techno', 'deep house'],
}

# ── global average distance (random sample of 5000 pairs) ────────────────────
np.random.seed(42)
N_SAMPLE = 5000
idx_a = np.random.randint(0, n_items, N_SAMPLE)
idx_b = np.random.randint(0, n_items, N_SAMPLE)
global_dists = [
    lorentz_distance(lorentz_table[i+1], lorentz_table[j+1])
    for i, j in zip(idx_a, idx_b) if i != j
]
global_avg = np.mean(global_dists)
print(f'Global average distance (random sample of {N_SAMPLE} pairs): {global_avg:.4f}')
print()

# ── intra-cluster average distances ──────────────────────────────────────────
rows = []
for cluster_name, genre_list in CLUSTERS_FOR_VALIDATION.items():
    found = [g for g in genre_list if g in genre_to_idx]
    missing = [g for g in genre_list if g not in genre_to_idx]
    if missing:
        print(f'  {cluster_name} — missing: {missing}')

    pairs = list(itertools.combinations(found, 2))
    if not pairs:
        continue

    dists = [
        lorentz_distance(get_embedding(g1), get_embedding(g2))
        for g1, g2 in pairs
        if get_embedding(g1) is not None and get_embedding(g2) is not None
    ]

    rows.append({
        'Cluster':            cluster_name,
        'Genres found':       len(found),
        'Pairs':              len(dists),
        'Intra-cluster mean': round(np.mean(dists), 4),
        'Intra-cluster std':  round(np.std(dists), 4),
        'vs Global avg':      'closer ✓' if np.mean(dists) < global_avg else 'further ✗'
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print(f'Global average distance: {global_avg:.4f}')

# ── bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
cluster_names = df['Cluster'].tolist()
intra_means   = df['Intra-cluster mean'].tolist()
colors = ['#e63946', '#f4a261', '#2a9d8f', '#8338ec', '#457b9d']

bars = ax.bar(cluster_names, intra_means, color=colors[:len(cluster_names)], edgecolor='white')
ax.axhline(global_avg, color='black', linewidth=2, linestyle='--', label=f'Global avg ({global_avg:.3f})')
ax.set_ylabel('Mean Lorentz Distance')
ax.set_title('Intra-cluster Distance vs Global Average')
ax.legend(fontsize=11)
ax.tick_params(axis='x', rotation=15)

# annotate bars
for bar, val in zip(bars, intra_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
# plt.savefig('intra_cluster_distances.png', dpi=150, bbox_inches='tight')  # uncomment to save
plt.show()